In [1]:
print("Hello VS Code, Python fonctionne !")

Hello VS Code, Python fonctionne !


In [2]:
!pip install praw pymongo pandas

   ---------------------------------------- 0.0/972.8 kB ? eta -:--:--
   ---------------------------------------- 972.8/972.8 kB 7.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------  9.7/9.8 MB 55.4 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 28.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ------------ --------------------------- 3.9/12.3 MB 64.7 MB/s eta 0:00:01
   ---------------------------------------  12.1/12.3 MB 64.6 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 31.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Uninstalling requests-2.32.3:
      Successfully uninstalled requests-2.32.3


In [1]:
import praw

In [3]:
!pip install --upgrade praw

  Attempting uninstall: praw
    Found existing installation: praw 8.0.0
    Uninstalling praw-8.0.0:
      Successfully uninstalled praw-8.0.0


In [48]:
reddit = praw.Reddit(
    client_id="6o5Pai7oa3CQhvbOO9z9SA",
    client_secret="_Eg7OvGzRvdTlqdZ4KFSjgjNkQT4BQ",
    user_agent="avis_app"
)

In [49]:
subreddit = reddit.subreddit("productreviews")

In [50]:
data = []

In [ ]:
for post in subreddit.hot(limit=1000):

_IncompleteInputError: incomplete input (394247696.py, line 1)

: 

In [51]:
for post in subreddit.hot(limit=1000):
    print(post.title)
    print(post.selftext)
    print("-----")

Introducing a new rewards system for honest, in-depth product reviews
There are no incentives for consumers to write honest in-depth reviews on platforms like Amazon. Why would you produce content for a retailer for free? Many sellers started giving out free products in exchange for positive reviews, which was the start of the whole fake reviews crisis.

People now come to Reddit for reviews because Redditors and other forum members are interested in boosting their ego by showing their depth of knowledge on the topic (and correcting others on the topic).

We're now announcing a new rewards system for high-quality reviews on our subreddit! This system is designed to incentivize users to write detailed, well-written reviews that provide valuable information to other users.

Here's how it will work:

* Users who write reviews will earn points based on the quality of their reviews. The points will be determined by a combination of factors such as the length and detail of the review, the ov

In [52]:
for post in subreddit.hot(limit=10000):
    data.append({
        "title": post.title,
        "text": post.selftext,
        "score": post.score
    })

In [53]:
print(len(data))

82


In [2]:
from pymongo import MongoClient

In [3]:
clean_data = []

for post in data:
    if post["title"] and post["text"] is not None:
        clean_data.append(post)

NameError: name 'data' is not defined

In [4]:
client = MongoClient("mongodb://localhost:27017/")
db = client["reddit_db"]
collection = db["reviews"]

In [5]:
result = collection.insert_many(clean_data)

print(len(result.inserted_ids))

TypeError: documents must be a non-empty list

In [58]:
for doc in collection.find():
    print(doc)

{'_id': ObjectId('6a31aeb5f18335f172f76285'), 'title': 'Introducing a new rewards system for honest, in-depth product reviews', 'text': "There are no incentives for consumers to write honest in-depth reviews on platforms like Amazon. Why would you produce content for a retailer for free? Many sellers started giving out free products in exchange for positive reviews, which was the start of the whole fake reviews crisis.\n\nPeople now come to Reddit for reviews because Redditors and other forum members are interested in boosting their ego by showing their depth of knowledge on the topic (and correcting others on the topic).\n\nWe're now announcing a new rewards system for high-quality reviews on our subreddit! This system is designed to incentivize users to write detailed, well-written reviews that provide valuable information to other users.\n\nHere's how it will work:\n\n* Users who write reviews will earn points based on the quality of their reviews. The points will be determined by a

In [6]:
collection.count_documents({})

82

In [9]:
import pandas as pd

data = list(collection.find())

df = pd.DataFrame(data)

df.head()

,_id,title,text,score
0,6a31aeb5f18335f172f76285,"Introducing a new rewards system for honest, i...",There are no incentives for consumers to write...,14
1,6a31aeb5f18335f172f76286,The Best Genmaicha Tea Brands of the Year,Genmaicha is a traditional Japanese tea that b...,3
2,6a31aeb5f18335f172f76287,First time drinking Rammstein Rum!,[https://youtu.be/Qx2c7V6gUNA](https://youtu.b...,3
3,6a31aeb5f18335f172f76288,IKEA Stenkol Battery Charger Review,I have been using my Duracell CEF14 battery ch...,3
4,6a31aeb5f18335f172f76289,Café Bustelo Coffee Review,,2


In [10]:
print(df.columns)

Index(['_id', 'title', 'text', 'score'], dtype='str')


In [11]:
df = df.drop(columns=["_id"])

In [12]:
df.head()

,title,text,score
0,"Introducing a new rewards system for honest, i...",There are no incentives for consumers to write...,14
1,The Best Genmaicha Tea Brands of the Year,Genmaicha is a traditional Japanese tea that b...,3
2,First time drinking Rammstein Rum!,[https://youtu.be/Qx2c7V6gUNA](https://youtu.b...,3
3,IKEA Stenkol Battery Charger Review,I have been using my Duracell CEF14 battery ch...,3
4,Café Bustelo Coffee Review,,2


In [13]:
print(df.isnull().sum())

title    0
text     0
score    0
dtype: int64


In [14]:
df["review"] = df["title"] + " " + df["text"]

In [15]:
print(df[["review"]].head())

                                              review
0  Introducing a new rewards system for honest, i...
1  The Best Genmaicha Tea Brands of the Year Genm...
2  First time drinking Rammstein Rum! [https://yo...
3  IKEA Stenkol Battery Charger Review I have bee...
4                        Café Bustelo Coffee Review 


In [16]:
df["sentiment"] = df["score"].apply(
    lambda x: 1 if x > 0 else 0
)

In [17]:
print(df["sentiment"].value_counts())

sentiment
1    73
0     9
Name: count, dtype: int64


In [18]:
df.to_csv("reddit_clean.csv", index=False)

In [23]:
import sqlite3

conn = sqlite3.connect("reddit_warehouse.db")

ERROR! Session/line number was not unique in database. History logging moved to new session 1171


In [24]:
df.to_sql(
    "reviews",
    conn,
    if_exists="replace",
    index=False
)

82

In [25]:
query = "SELECT * FROM reviews LIMIT 5"

test = pd.read_sql(query, conn)

print(test)

                                               title  \
0  Introducing a new rewards system for honest, i...   
1          The Best Genmaicha Tea Brands of the Year   
2                 First time drinking Rammstein Rum!   
3                IKEA Stenkol Battery Charger Review   
4                         Café Bustelo Coffee Review   

                                                text  score  \
0  There are no incentives for consumers to write...     14   
1  Genmaicha is a traditional Japanese tea that b...      3   
2  [https://youtu.be/Qx2c7V6gUNA](https://youtu.b...      3   
3  I have been using my Duracell CEF14 battery ch...      3   
4                                                         2   

                                              review  sentiment  
0  Introducing a new rewards system for honest, i...          1  
1  The Best Genmaicha Tea Brands of the Year Genm...          1  
2  First time drinking Rammstein Rum! [https://yo...          1  
3  IKEA Stenkol Batt

In [22]:
!pip install google-cloud-bigquery
!pip install pandas-gbq
!pip install pyarrow

   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.8 MB 723.9 kB/s eta 0:00:05
   -------- ------------------------------- 0.8/3.8 MB 749.3 kB/s eta 0:00:05
   -------- ------------------------------- 0.8/3.8 MB 749.3 kB/s eta 0:00:05
   ----------- ---------------------------- 1.0/3.8 MB 749.7 kB/s eta 0:00:04
   ------------- -------------------------- 1.3/3.8 MB 765.1 kB/s eta 0:00:04
   ------------- -------------------------- 1.3/3.8 MB 765.1 kB/s eta 0:00:04
   ---------------- ----------------------- 1.6/3.8 MB 767.1 kB/s eta 0:00:03
   ---------------- ----------------------- 1.6/3.8 MB 767.1 kB/s eta 0:00:03
   ------------------- -------------------- 1.8/3.8 MB 756.4 kB/s eta 0:00:03
   -------------------

In [26]:
df = pd.read_sql("SELECT * FROM reviews", conn)

print(df.shape)
print(df.head())

(82, 5)
                                               title  \
0  Introducing a new rewards system for honest, i...   
1          The Best Genmaicha Tea Brands of the Year   
2                 First time drinking Rammstein Rum!   
3                IKEA Stenkol Battery Charger Review   
4                         Café Bustelo Coffee Review   

                                                text  score  \
0  There are no incentives for consumers to write...     14   
1  Genmaicha is a traditional Japanese tea that b...      3   
2  [https://youtu.be/Qx2c7V6gUNA](https://youtu.b...      3   
3  I have been using my Duracell CEF14 battery ch...      3   
4                                                         2   

                                              review  sentiment  
0  Introducing a new rewards system for honest, i...          1  
1  The Best Genmaicha Tea Brands of the Year Genm...          1  
2  First time drinking Rammstein Rum! [https://yo...          1  
3  IKEA Sten

In [27]:
print(df.columns)

Index(['title', 'text', 'score', 'review', 'sentiment'], dtype='str')


In [28]:
df["review"] = df["review"].fillna("")
df = df.dropna()

In [29]:
X = df["review"]
y = df["sentiment"]

In [31]:
!pip install scikit-learn

^C


  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 671.5 kB/s eta 0:00:12
   -- ------------------------------------- 0.5/8.2 MB 671.5 kB/s eta 0:00:12
   --- ------------------------------------ 0.8/8.2 MB 716.8 kB/s eta 0:00:11
   ----- ---------------------------------- 1.0/8.2 MB 715.9 kB/s eta 0:00:10
   ----- ---------------------------------- 1.0/8.2 MB 715.9 kB/s eta 0:00:10
   ------ --------------------------------- 1.3/8.2 MB 711.1 kB/s eta 0:00:10
   ------ --------------------------------- 1.3/8.2 MB 711.1 kB/s eta 0:00:10
   ------- -------------------------------- 1.6/8.2 MB 717.6 kB/s eta 0:00:10
   -------- -------------------

In [32]:
import sklearn
print(sklearn.__version__)

1.9.0


In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

ÉTAPE 1 — TF-IDF (vectorisation du texte)

Transformer le texte en chiffres (obligatoire pour ML)

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer

2. Créer le vectorizer

In [36]:
vectorizer = TfidfVectorizer()

3. Apprendre + transformer le train set

In [37]:
X_train_vec = vectorizer.fit_transform(X_train)

In [ ]:
4. Transformer le test set

In [38]:
X_test_vec = vectorizer.transform(X_test)

5. Vérification

In [39]:
print(X_train_vec.shape)
print(X_test_vec.shape)

(65, 944)
(17, 944)


In [40]:
from sklearn.linear_model import LogisticRegression

In [41]:
model = LogisticRegression()

In [42]:
model.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [43]:
y_pred = model.predict(X_test_vec)

In [44]:
print(y_pred[:10])

[1 1 1 1 1 1 1 1 1 1]


In [45]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9411764705882353


In [46]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.94      1.00      0.97        16

    accuracy                           0.94        17
   macro avg       0.47      0.50      0.48        17
weighted avg       0.89      0.94      0.91        17



c:\Users\sh_mi\anaconda3\envs\notebook\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sh_mi\anaconda3\envs\notebook\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sh_mi\anaconda3\envs\notebook\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

In [58]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

['vectorizer.pkl']

In [59]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib

# données exemple
texts = ["j'aime ce produit", "je déteste ce produit"]
labels = [1, 0]

# vectorisation
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

# modèle
model = LogisticRegression()
model.fit(X, labels)

# sauvegarde
joblib.dump(model, "model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("Modèle et vectorizer sauvegardés !")

Modèle et vectorizer sauvegardés !


In [60]:
print(model)
print(vectorizer)

LogisticRegression()
TfidfVectorizer()


In [61]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

['vectorizer.pkl']

In [63]:
import joblib

joblib.dump(model, "model.pkl")

['model.pkl']

In [62]:
import os

print(os.getcwd())

c:\Users\sh_mi\AppData\Local\Programs\Microsoft VS Code


In [49]:
loaded_model = joblib.load("model.pkl")
loaded_vectorizer = joblib.load("vectorizer.pkl")

text = ["this product is very bad"]

vec = loaded_vectorizer.transform(text)
print(loaded_model.predict(vec))

[1]


In [50]:
!pip install fastapi uvicorn

  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 844.5 kB/s eta 0:00:02
   ---------- ----------------------------- 0.5/2.1 MB 844.5 kB/s eta 0:00:02
   ---------- ----------------------------- 0.5/2.1 MB 844.5 kB/s eta 0:00:02
   ---------- ----------------------------- 0.5/2.1 MB 844.5 kB/s eta 0:00:02
   ---------- ----------------------------- 0.5/2.1 MB 844.5 kB/s eta 0:00:02
   --------------- ------------------------ 0.8/2.1 MB 400.7 kB/s eta 0:00:04
   --------------- -----------------

fichier : main.py

In [56]:
!pip install fastapi
import fastapi

In [51]:
from fastapi import FastAPI
import joblib

app = FastAPI()

model = joblib.load("model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

Endpoint prediction

In [52]:
@app.post("/predict")
def predict(text: str):
    vec = vectorizer.transform([text])
    pred = model.predict(vec)

    sentiment = "positif" if pred[0] == 1 else "negatif"

    return {"sentiment": sentiment}

Lancer

In [53]:
print("MAIN FILE LOADED")



MAIN FILE LOADED


In [57]:
import joblib

model = joblib.load("model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

In [66]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib

# mini dataset
texts = [
    "j'aime ce produit",
    "je déteste ce produit",
    "c'est super",
    "c'est nul"
]

labels = [1, 0, 1, 0]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

model = LogisticRegression()
model.fit(X, labels)

joblib.dump(model, "model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("OK modèles recréés")

OK modèles recréés
